In [6]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-b8e7c0a0-6e77-ee51-bec8-f73dc2d83050, 7.5, 15360 MiB
head: option requires an argument -- 'n'
Try 'head --help' for more information.


In [7]:
GITHUB_USERNAME = "beIover"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, \
    "You must set GITHUB_USERNAME to your real GitHub handle."
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: beIover


In [12]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import time
import math
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8"
    ).strip().splitlines()[0].strip()
    return out

def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)

# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    dims = MPI.Compute_dims(size, [2, 2])
    cart_comm = comm.Create_cart(dims, periods=[False, False], reorder=False)

    up_nbr, down_nbr = cart_comm.Shift(0, 1)
    left_nbr, right_nbr = cart_comm.Shift(1, 1)
    rank_row, rank_col = cart_comm.Get_coords(rank)

    return cart_comm, rank_row, rank_col, up_nbr, down_nbr, left_nbr, right_nbr, instance_seed, gpu_uuid

# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ==============================================================================
def initialize_subdomains(rank_row, rank_col, seed):
    nx, ny = GLOBAL_NX // 2, GLOBAL_NY // 2
    u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    kappa = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    S = np.zeros((ny + 2, nx + 2), dtype=np.float64)

    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)

    # Детерминированная генерация параметров из seed
    rng = np.random.default_rng(seed)
    c1_x, c1_y = rng.uniform(0.25, 0.75, 2)
    c2_x, c2_y = rng.uniform(0.25, 0.75, 2)
    amp1, amp2 = rng.uniform(0.5, 1.5, 2)

    for r in range(ny + 2):
        gy = rank_row * ny + r
        y = gy * dy
        for c in range(nx + 2):
            gx = rank_col * nx + c
            x = gx * dx

            # Пространственно неоднородная теплопроводность kappa in [0.08, 0.24]
            val_k = 0.16 + 0.08 * math.sin(2.0 * math.pi * x) * math.cos(2.0 * math.pi * y)
            kappa[r, c] = min(max(val_k, 0.08), 0.24)

            # Начальные гауссовы импульсы и источник S(x,y)
            d1 = (x - c1_x)**2 + (y - c1_y)**2
            d2 = (x - c2_x)**2 + (y - c2_y)**2
            u[r, c] = amp1 * math.exp(-d1 / 0.01) + amp2 * math.exp(-d2 / 0.015)
            S[r, c] = 0.5 * amp1 * math.exp(-d1 / 0.005)

    # Граничные условия Дирихле (u = 0 на внешних границах)
    if rank_row == 0:
        u[0, :] = 0.0
    if rank_row == 1:
        u[ny + 1, :] = 0.0
    if rank_col == 0:
        u[:, 0] = 0.0
    if rank_col == 1:
        u[:, nx + 1] = 0.0

    return (
        cuda.to_device(u),
        cuda.to_device(u.copy()),
        cuda.to_device(kappa),
        cuda.to_device(S),
        float(np.max(kappa))
    )

# ==============================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels
# ==============================================================================
@cuda.jit
def pack_halo_kernel(u, top, bottom, left, right, nx, ny):
    idx = cuda.grid(1)
    if idx < nx:
        top[idx] = u[1, idx + 1]
        bottom[idx] = u[ny, idx + 1]
    if idx < ny:
        left[idx] = u[idx + 1, 1]
        right[idx] = u[idx + 1, nx]

@cuda.jit
def unpack_halo_kernel(u, top, bottom, left, right, nx, ny):
    idx = cuda.grid(1)
    if idx < nx:
        u[0, idx + 1] = top[idx]
        u[ny + 1, idx + 1] = bottom[idx]
    if idx < ny:
        u[idx + 1, 0] = left[idx]
        u[idx + 1, nx + 1] = right[idx]

# ==============================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil Kernel
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_next, kappa, S, r_start, r_end, c_start, c_end, dx, dy, dt):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    bx = cuda.blockIdx.x
    by = cuda.blockIdx.y

    i = by * TILE_DIM + ty + 1
    j = bx * TILE_DIM + tx + 1

    shmem = cuda.shared.array((18, 18), dtype=float64)

    # Загрузка центральных ячеек и 1-ячеечного гало
    if i <= 256 and j <= 256:
        shmem[ty + 1, tx + 1] = u[i, j]
        if tx == 0:
            shmem[ty + 1, 0] = u[i, j - 1]
        if tx == TILE_DIM - 1:
            shmem[ty + 1, TILE_DIM + 1] = u[i, j + 1]
        if ty == 0:
            shmem[0, tx + 1] = u[i - 1, j]
        if ty == TILE_DIM - 1:
            shmem[TILE_DIM + 1, tx + 1] = u[i + 1, j]

    cuda.syncthreads()

    # Вычисление только внутри заданного bounding-box
    if r_start <= i <= r_end and c_start <= j <= c_end:
        u_c = shmem[ty + 1, tx + 1]
        u_l = shmem[ty + 1, tx]
        u_r = shmem[ty + 1, tx + 2]
        u_u = shmem[ty, tx + 1]
        u_d = shmem[ty + 2, tx + 1]

        k_c = kappa[i, j]
        k_iph = 0.5 * (k_c + kappa[i, j + 1])
        k_imh = 0.5 * (k_c + kappa[i, j - 1])
        k_jph = 0.5 * (k_c + kappa[i + 1, j])
        k_jmh = 0.5 * (k_c + kappa[i - 1, j])

        flux_x = (k_iph * (u_r - u_c) - k_imh * (u_c - u_l)) / (dx * dx)
        flux_y = (k_jph * (u_d - u_c) - k_jmh * (u_c - u_u)) / (dy * dy)

        u_next[i, j] = u_c + dt * (flux_x + flux_y - LAMBDA_DAMP * (u_c**3) + S[i, j])

# ==============================================================================
# TASK 5: Dual-Stream Orchestrator & Comparative Benchmark
# ==============================================================================
def run_solver(cart_comm, u_init, kappa_d, S_d, nbrs, dx, dy, dt, overlapped=True):
    up, down, left, right = nbrs
    nx, ny = 256, 256
    u_d = cuda.to_device(u_init.copy_to_host())
    u_next_d = cuda.to_device(u_init.copy_to_host())

    # Буферы на GPU и pinned-буферы на CPU
    d_send_top, d_send_bot = cuda.device_array(nx, dtype=np.float64), cuda.device_array(nx, dtype=np.float64)
    d_send_left, d_send_right = cuda.device_array(ny, dtype=np.float64), cuda.device_array(ny, dtype=np.float64)
    d_recv_top, d_recv_bot = cuda.device_array(nx, dtype=np.float64), cuda.device_array(nx, dtype=np.float64)
    d_recv_left, d_recv_right = cuda.device_array(ny, dtype=np.float64), cuda.device_array(ny, dtype=np.float64)

    h_send_top, h_send_bot = cuda.pinned_array(nx, dtype=np.float64), cuda.pinned_array(nx, dtype=np.float64)
    h_send_left, h_send_right = cuda.pinned_array(ny, dtype=np.float64), cuda.pinned_array(ny, dtype=np.float64)
    h_recv_top, h_recv_bot = cuda.pinned_array(nx, dtype=np.float64), cuda.pinned_array(nx, dtype=np.float64)
    h_recv_left, h_recv_right = cuda.pinned_array(ny, dtype=np.float64), cuda.pinned_array(ny, dtype=np.float64)

    stream_int = cuda.stream()
    stream_bnd = cuda.stream() if overlapped else stream_int

    grid_2d = (16, 16)
    block_2d = (16, 16)

    cart_comm.Barrier()
    t0 = time.perf_counter()

    for step in range(STEPS):
        # Stage A: Pack & Async D2H
        pack_halo_kernel[4, 64, stream_bnd](u_d, d_send_top, d_send_bot, d_send_left, d_send_right, nx, ny)
        d_send_top.copy_to_host(h_send_top, stream=stream_bnd)
        d_send_bot.copy_to_host(h_send_bot, stream=stream_bnd)
        d_send_left.copy_to_host(h_send_left, stream=stream_bnd)
        d_send_right.copy_to_host(h_send_right, stream=stream_bnd)

        if overlapped:
            # Stage B: Non-blocking MPI + Concurrent Interior Compute
            stream_bnd.synchronize()
            reqs = [
                cart_comm.Irecv(h_recv_top, source=up, tag=1),
                cart_comm.Irecv(h_recv_bot, source=down, tag=0),
                cart_comm.Irecv(h_recv_left, source=left, tag=3),
                cart_comm.Irecv(h_recv_right, source=right, tag=2),
                cart_comm.Isend(h_send_top, dest=up, tag=0),
                cart_comm.Isend(h_send_bot, dest=down, tag=1),
                cart_comm.Isend(h_send_left, dest=left, tag=2),
                cart_comm.Isend(h_send_right, dest=right, tag=3),
            ]
            # Считаем глубокую внутреннюю область [2..255, 2..255] пока идет обмен по сети
            tiled_stencil_kernel[grid_2d, block_2d, stream_int](u_d, u_next_d, kappa_d, S_d, 2, 255, 2, 255, dx, dy, dt)

            # Stage C: Wait MPI, Async H2D & Boundary Ring Stencil
            MPI.Request.Waitall(reqs)
            if up != MPI.PROC_NULL: d_recv_top.copy_to_device(h_recv_top, stream=stream_bnd)
            if down != MPI.PROC_NULL: d_recv_bot.copy_to_device(h_recv_bot, stream=stream_bnd)
            if left != MPI.PROC_NULL: d_recv_left.copy_to_device(h_recv_left, stream=stream_bnd)
            if right != MPI.PROC_NULL: d_recv_right.copy_to_device(h_recv_right, stream=stream_bnd)

            unpack_halo_kernel[4, 64, stream_bnd](u_d, d_recv_top, d_recv_bot, d_recv_left, d_recv_right, nx, ny)

            # Восстановление внешних границ Дирихле (u=0)
            if up == MPI.PROC_NULL: u_d[0, :] = 0.0
            if down == MPI.PROC_NULL: u_d[ny+1, :] = 0.0
            if left == MPI.PROC_NULL: u_d[:, 0] = 0.0
            if right == MPI.PROC_NULL: u_d[:, nx+1] = 0.0

            # Обновляем 4 граничных кольца
            tiled_stencil_kernel[grid_2d, block_2d, stream_bnd](u_d, u_next_d, kappa_d, S_d, 1, 1, 1, 256, dx, dy, dt)
            tiled_stencil_kernel[grid_2d, block_2d, stream_bnd](u_d, u_next_d, kappa_d, S_d, 256, 256, 1, 256, dx, dy, dt)
            tiled_stencil_kernel[grid_2d, block_2d, stream_bnd](u_d, u_next_d, kappa_d, S_d, 2, 255, 1, 1, dx, dy, dt)
            tiled_stencil_kernel[grid_2d, block_2d, stream_bnd](u_d, u_next_d, kappa_d, S_d, 2, 255, 256, 256, dx, dy, dt)

            stream_int.synchronize()
            stream_bnd.synchronize()
        else:
            # Serialized режим (без перекрытия)
            stream_bnd.synchronize()
            reqs = [
                cart_comm.Irecv(h_recv_top, source=up, tag=1),
                cart_comm.Irecv(h_recv_bot, source=down, tag=0),
                cart_comm.Irecv(h_recv_left, source=left, tag=3),
                cart_comm.Irecv(h_recv_right, source=right, tag=2),
                cart_comm.Isend(h_send_top, dest=up, tag=0),
                cart_comm.Isend(h_send_bot, dest=down, tag=1),
                cart_comm.Isend(h_send_left, dest=left, tag=2),
                cart_comm.Isend(h_send_right, dest=right, tag=3),
            ]
            MPI.Request.Waitall(reqs)
            if up != MPI.PROC_NULL: d_recv_top.copy_to_device(h_recv_top, stream=stream_bnd)
            if down != MPI.PROC_NULL: d_recv_bot.copy_to_device(h_recv_bot, stream=stream_bnd)
            if left != MPI.PROC_NULL: d_recv_left.copy_to_device(h_recv_left, stream=stream_bnd)
            if right != MPI.PROC_NULL: d_recv_right.copy_to_device(h_recv_right, stream=stream_bnd)
            unpack_halo_kernel[4, 64, stream_bnd](u_d, d_recv_top, d_recv_bot, d_recv_left, d_recv_right, nx, ny)
            if up == MPI.PROC_NULL: u_d[0, :] = 0.0
            if down == MPI.PROC_NULL: u_d[ny+1, :] = 0.0
            if left == MPI.PROC_NULL: u_d[:, 0] = 0.0
            if right == MPI.PROC_NULL: u_d[:, nx+1] = 0.0
            tiled_stencil_kernel[grid_2d, block_2d, stream_bnd](u_d, u_next_d, kappa_d, S_d, 1, 256, 1, 256, dx, dy, dt)
            stream_bnd.synchronize()

        u_d, u_next_d = u_next_d, u_d

        if step % 100 == 0 or step == STEPS - 1:
            host_u = u_d.copy_to_host()[1:ny+1, 1:nx+1]
            local_energy = float(np.sum(host_u) * dx * dy)
            global_energy = cart_comm.allreduce(local_energy, op=MPI.SUM)
            if cart_comm.Get_rank() == 0 and overlapped and (step % 100 == 0):
                print(f"[Overlapped] Step {step:03d} | Global Thermal Energy E = {global_energy:.8f}")

    cart_comm.Barrier()
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    return u_d.copy_to_host()[1:ny+1, 1:nx+1], elapsed_ms

if __name__ == "__main__":
    github_user = sys.argv[1]
    cart_comm, r_row, r_col, up, down, left, right, seed, gpu_uuid = init_distributed_context(github_user)
    rank = cart_comm.Get_rank()

    u_init, _, kappa_d, S_d, local_k_max = initialize_subdomains(r_row, r_col, seed)
    global_k_max = cart_comm.allreduce(local_k_max, op=MPI.MAX)

    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    dt_max = (dx * dx) / (4.0 * global_k_max)
    dt = 0.8 * dt_max  # Безопасный шаг с запасом по CFL

    nbrs = (up, down, left, right)

    # Прогрев JIT-компиляции Numba
    _ = run_solver(cart_comm, u_init, kappa_d, S_d, nbrs, dx, dy, dt, overlapped=True)

    # Основные прогоны
    state_ovl, overlapped_ms = run_solver(cart_comm, u_init, kappa_d, S_d, nbrs, dx, dy, dt, overlapped=True)
    state_ser, serialized_ms = run_solver(cart_comm, u_init, kappa_d, S_d, nbrs, dx, dy, dt, overlapped=False)

    local_diff = float(np.max(np.abs(state_ovl - state_ser)))
    parity_max_diff = cart_comm.allreduce(local_diff, op=MPI.MAX)

    # Аудит PTX инструкций ядра
    ptx_dict = tiled_stencil_kernel.inspect_asm()
    ptx_code = next(iter(ptx_dict.values()))
    st_shared_cnt = ptx_code.count("st.shared")
    ld_shared_cnt = ptx_code.count("ld.shared")
    bar_sync_cnt = ptx_code.count("bar.sync")
    ptx_passed = (st_shared_cnt > 0) and (ld_shared_cnt > 0) and (bar_sync_cnt > 0)

    if rank == 0:
        print("\n=== BENCHMARK & AUDIT SUMMARY ===")
        print(f"MPI Ranks: 4 (2x2 Cartesian Grid)")
        print(f"GPU UUID: {gpu_uuid} | Seed: {seed}")
        print(f"Max Diffusivity (kappa_max): {global_k_max:.6f} | dt_max: {dt_max:.4e} | dt_used: {dt:.4e}")
        print(f"Serialized Time: {serialized_ms:.2f} ms | Overlapped Time: {overlapped_ms:.2f} ms")
        print(f"Parity Max Diff: {parity_max_diff:.4e} (Valid: {parity_max_diff < 1e-12})")
        print(f"PTX Counts -> st.shared: {st_shared_cnt}, ld.shared: {ld_shared_cnt}, bar.sync: {bar_sync_cnt}")

        manifest = {
            "GITHUB_USERNAME": github_user,
            "gpu_uuid": gpu_uuid,
            "instance_seed": seed,
            "mpi_ranks": 4,
            "parity_max_diff": parity_max_diff,
            "ptx_audit_passed": ptx_passed,
            "ptx_counts": {
                "st.shared": st_shared_cnt,
                "ld.shared": ld_shared_cnt,
                "bar.sync": bar_sync_cnt
            },
            "metrics": {
                "serialized_ms": serialized_ms,
                "overlapped_ms": overlapped_ms,
                "kappa_max": global_k_max,
                "dt_max": dt_max,
                "dt_used": dt
            }
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("Generated submission_manifest.json successfully!")

Overwriting distributed_cuda_solver.py


In [9]:
!apt-get update -qq
!apt-get install -y -qq openmpi-bin libopenmpi-dev
!pip install -q mpi4py numba numpy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [13]:
import subprocess
cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME
]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")
print(proc.stdout)

[Overlapped] Step 000 | Global Thermal Energy E = 0.09714317
[Overlapped] Step 100 | Global Thermal Energy E = 0.09713989
[Overlapped] Step 200 | Global Thermal Energy E = 0.09713760
[Overlapped] Step 300 | Global Thermal Energy E = 0.09713541
[Overlapped] Step 000 | Global Thermal Energy E = 0.09714317
[Overlapped] Step 100 | Global Thermal Energy E = 0.09713989
[Overlapped] Step 200 | Global Thermal Energy E = 0.09713760
[Overlapped] Step 300 | Global Thermal Energy E = 0.09713541

=== BENCHMARK & AUDIT SUMMARY ===
MPI Ranks: 4 (2x2 Cartesian Grid)
GPU UUID: GPU-b8e7c0a0-6e77-ee51-bec8-f73dc2d83050 | Seed: 9494468596201668792
Max Diffusivity (kappa_max): 0.240000 | dt_max: 3.9582e-06 | dt_used: 3.1665e-06
Serialized Time: 2682.70 ms | Overlapped Time: 3086.89 ms
Parity Max Diff: 0.0000e+00 (Valid: True)
PTX Counts -> st.shared: 5, ld.shared: 5, bar.sync: 1
Generated submission_manifest.json successfully!



### Analytical & Forensic Engineering Responses

- **Question 1 (Memory Coalescing & Strided Column Packing):**
In a C-contiguous (row-major) `float64` device array of shape `(258, 258)`, vertical halo columns (`u[1:257, 1]` and `u[1:257, 256]`) are non-contiguous in physical memory, separated by a row stride of `258 * 8 = 2064` bytes. A PCIe DMA engine requires a contiguous memory block for a single transfer. Attempting an unstrided bounding-box copy directly from `u[1, 1]` to `u[256, 256]` to transfer a single column would force PCIe to transfer the entire spanned sub-matrix (`256 * 258 * 8 = 528,384` bytes) instead of just the `256 * 8 = 2,048` bytes of the column—wasting **526,336 extra bytes** per column transfer (or **1,052,672 extra bytes** for both left and right columns). Dedicated 1D packing kernels coalesce these strided reads into contiguous buffers on the GPU first.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
In the `18 x 18` `float64` shared-memory tile, local thread `(tx=0, ty=0)` reads its center value from `shmem[1, 1]` (linear index `1*18 + 1 = 19`) and neighbors at `shmem[1, 0]`, `shmem[1, 2]`, `shmem[0, 1]`, and `shmem[2, 1]`. Thread `(tx=15, ty=15)` reads its center from `shmem[16, 16]` (linear index `16*18 + 16 = 304`) and neighbors at `shmem[16, 15]`, `shmem[16, 17]`, `shmem[15, 16]`, and `shmem[17, 16]`. On NVIDIA Compute Capability 7.5 (Tesla T4), shared memory has 32 banks of 4 bytes each (128 bytes per cycle). Because `float64` is 8 bytes, every 64-bit load spans two consecutive banks, inherently causing a **2-way bank conflict** across a 32-thread warp when accessing row-major elements (`16 threads * 8 bytes = 128 bytes` per half-warp). Furthermore, each row of 18 `float64` elements is `144` bytes (`144 mod 128 = 16` bytes, or 4 banks), which shifts the bank alignment of consecutive rows by 4 banks during vertical column accesses.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
From the PTX inspection of `tiled_stencil_kernel` in this run, the exact instruction counts emitted are **`st.shared`: 5**, **`ld.shared`: 5**, and **`bar.sync`: 1**. A `16 x 16` threadblock consists of `256 / 32 = 8` warps. If `cuda.syncthreads()` (`bar.sync`) is omitted between loading the `18 x 18` halo apron and computing the stencil, a cross-warp **Read-After-Write (RAW) memory hazard** occurs: fast warps proceed to execute `ld.shared` for neighbor cells before slower boundary warps have completed their `st.shared` writes of the 1-cell halo apron, resulting in race conditions and corrupted spatial derivatives.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
In this run, the measured wall-clock execution times were **`serialized_ms = 2682.70 ms`** versus **`overlapped_ms = 3086.89 ms`**. When `mpirun -np 4` launches four OS processes on a single Tesla T4 GPU, the CUDA driver arbitrates access via hardware context time-slicing across the shared Streaming Multiprocessors (SMs) and single PCIe DMA copy engine. Instead of true hardware concurrency, the four ranks serialize at the GPU command queue and PCIe bus, while dual-stream orchestration adds extra kernel launch and synchronization overhead (`404.19 ms` overhead here). On a multi-node cluster with four dedicated GPUs over InfiniBand, each rank owns independent SMs and dedicated DMA/NIC engines, allowing genuine overlap of interior compute with inter-node halo transfers.

- **Question 5 (Point-to-Point Message Deadlock & MPI.PROC_NULL):**
If `Isend`/`Irecv` are replaced with blocking `Send` followed by blocking `Recv` in a fixed directional order across all ranks, circular deadlock occurs once message sizes exceed MPI's internal **eager protocol threshold** (typically a few kilobytes). Below this threshold, `MPI.Send` copies data into an internal system buffer and returns immediately; once grid dimensions grow large enough that the halo vector exceeds the eager limit, MPI switches to the **rendezvous protocol**, where `Send` blocks until the target rank posts a matching `Recv`. Because all ranks block inside `Send` simultaneously, no rank ever reaches `Recv`. Using `MPI.PROC_NULL` for out-of-bounds neighbors simplifies boundary logic on corner ranks like `(0, 0)` because `Isend`/`Irecv` calls targeting `MPI.PROC_NULL` act as immediate no-ops, eliminating conditional `if/else` branching around communication calls.

- **Question 6 (Explicit CFL Stability & Energy Conservation):**
Using the recorded maximum diffusivity **`kappa_max = 0.240000`** and grid spacing $\Delta x = 1/513$, the 2D explicit diffusion stability limit is:
$$\Delta t_{\max} = \frac{\Delta x^2}{4 \cdot \kappa_{\max}} = \frac{(1/513)^2}{4 \cdot 0.24} = 3.9582 \times 10^{-6}$$
Our simulation used **`dt_used = 3.1665e-06`** ($0.8 \cdot \Delta t_{\max} < \Delta t_{\max}$), strictly satisfying the CFL condition. As verified by the `MPI.SUM` `allreduce` telemetry across steps, the global thermal energy $E$ exhibits strictly monotonic dissipative decay without numerical blow-up:
- **Step 000:** `E = 0.09714317`
- **Step 100:** `E = 0.09713989`
- **Step 200:** `E = 0.09713760`
- **Step 300:** `E = 0.09713541`